# How to Load Tabular Data

The fastest way to get music data into Time To Align! is through **tabular
loaders**. If your data is in CSV or TSV format, you're just 3 lines of
code away from analysis.

**What you'll learn:**
- Load music annotations from TSV/CSV files
- Access event counts, coordinate ranges, and metadata
- Create timelines from loaded data
- Write a custom loader that maps your columns to event fields
- Promote selected columns to typed fields with `column_specs`
- Reach nested JSON columns with `Field`

**Time:** 15 minutes

This is the gentle introduction. For the full column-to-field mechanism —
the Step-1 resolution chain, composite columns, and Step-2 field
promotion — see the in-depth CSV/TSV how-to.

## TL;DR

```python
from timetoalign.loader.tabular import Ms3Loader

loader = Ms3Loader()
loader.load("beethoven.notes.tsv")

df = loader.events.to_pandas()       # Get as DataFrame
timeline = loader.create_timeline()  # Create Timeline
```

## Setup

In [1]:
from timetoalign.testdata import ensure_data

BEETHOVEN = ensure_data("score") / "beethoven_woo71"
THORESEN = ensure_data("thoresen")

# Available files
{
    "Beethoven files": [f.name for f in BEETHOVEN.glob("WoO71.*.tsv")],
    "Thoresen files": [f.name for f in THORESEN.glob("*.tsv")],
}

{'Beethoven files': ['WoO71.chords.tsv',
  'WoO71.measures.tsv',
  'WoO71.notes.tsv'],
 'Thoresen files': ['thoresen_test_h.tsv', 'thoresen_test.tsv']}

## Loading Notes from TSV

The `Ms3Loader` handles TSV files exported from the ms3 parser, which
processes MuseScore files.

**Three lines of code:**

In [2]:
from timetoalign.loader.tabular import Ms3Loader  # noqa: E402

loader = Ms3Loader()
loader.load(BEETHOVEN / "WoO71.notes.tsv")

f"{len(loader.events):,} notes loaded"

'4,753 notes loaded'

## Converting to pandas

Use `to_pandas()` to get a DataFrame with clean coordinate values:

In [3]:
loader.events.to_pandas().head()

,id,name,temporal_type,event_type,start,end,duration,mc,midi,mn,...,mn_onset,voice,staff,gracenote,tpc,timesig,nominal_duration,chord_id,quarterbeats,octave
0,e000000,A3,interval,Note,0,1/4,1/4,1,57,0,...,1/4,1,2,NaN,3,2/4,1/4,3,0,3
1,e000001,E4,interval,Note,0,1/4,1/4,1,64,0,...,1/4,2,1,NaN,4,2/4,1/4,2,0,4
2,e000002,A4,interval,Note,0,1/8,1/8,1,69,0,...,1/4,1,1,NaN,3,2/4,1/8,0,0,4
3,e000003,C#5,interval,Note,0,1/8,1/8,1,73,0,...,1/4,1,1,NaN,7,2/4,1/8,0,0,5
4,e000004,E5,interval,Note,1/2,5/8,1/8,1,76,0,...,3/8,1,1,NaN,4,2/4,1/8,1,1/2,5


## Quick Statistics

The loader provides immediate access to summary information:

In [4]:
{
    "event_count": len(loader.events),
    "coordinate_range": loader.events.coordinate_range(),
    "unit": str(loader.unit),
    "number_type": str(loader.number_type),
}

{'event_count': 4753,
 'coordinate_range': (Fraction(0, 1), Fraction(3511, 4)),
 'unit': 'quarters',
 'number_type': 'fraction'}

## Creating Timelines

Time To Align! represents temporal data as **Timelines**:

In [5]:
timeline = loader.create_timeline(uid="beethoven_notes")
timeline

ContinuousLogicalTimeline(id='beethoven_notes', length=3511/4, unit=quarters, events=4753, children=0)

## Custom Loaders for Non-Standard Formats

For files that don't match the ms3 format, create a custom loader by
subclassing `TsvLoader` or `CsvLoader`. You point the canonical column
attributes (`start_column`, `duration_column`, …) at the names in your
file.

Let's load the Thoresen annotations file, which has a different column
structure:

In [6]:
import pandas as pd  # noqa: E402

pd.read_csv(THORESEN / "thoresen_test.tsv", sep="\t", nrows=3)

,event_id,alignment_group_id,start_time_sec,duration_sec,event_type,graphical_element_id,image_filename,rect_coords_json,text_content,text_anchor_xy_json,layer_order,description
0,annot_cue_001,NaN,0.0,5.0,rectangle,rect_a,thoresen_2010_form-building-patterns_p90-91_pa...,"{""x"": 10, ""y"": 90, ""width"": 148, ""height"": 55}",NaN,NaN,NaN,NaN
1,annot_cue_002,NaN,1.5,4.0,rectangle,rect_b,thoresen_2010_form-building-patterns_p90-91_pa...,"{""x"": 40, ""y"": 37, ""width"": 127, ""height"": 21}",NaN,NaN,NaN,NaN
2,annot_cue_003,NaN,3.5,2.0,rectangle,rect_c,thoresen_2010_form-building-patterns_p90-91_pa...,"{""x"": 111, ""y"": 60, ""width"": 57, ""height"": 23}",NaN,NaN,NaN,NaN


### The Simplest Custom Loader

Map the core coordinate columns and nothing else. Every source column
that you do *not* name survives as an opaque **property column** —
carried alongside the event so you never lose data, but left untyped:

In [7]:
from timetoalign.core import NumberType, TimeUnit  # noqa: E402
from timetoalign.loader.tabular import TsvLoader  # noqa: E402


class ThoresenLoader(TsvLoader):
    """Minimal loader — maps the core event fields."""

    id_column = "event_id"
    start_column = "start_time_sec"
    duration_column = "duration_sec"
    event_type_column = "event_type"
    name_column = "description"

    _default_unit = TimeUnit.seconds
    coordinate_type = NumberType.float


thoresen = ThoresenLoader()
thoresen.load(THORESEN / "thoresen_test.tsv")
thoresen.events.to_pandas()

,id,name,temporal_type,event_type,start,end,duration,text_anchor_xy_json,alignment_group_id,image_filename,text_content,rect_coords_json,layer_order,graphical_element_id
0,annot_cue_001,NaN,interval,rectangle,0.0,5.00,5.00,NaN,NaN,thoresen_2010_form-building-patterns_p90-91_pa...,NaN,"{""x"": 10, ""y"": 90, ""width"": 148, ""height"": 55}",NaN,rect_a
1,annot_cue_002,NaN,interval,rectangle,1.5,5.50,4.00,NaN,NaN,thoresen_2010_form-building-patterns_p90-91_pa...,NaN,"{""x"": 40, ""y"": 37, ""width"": 127, ""height"": 21}",NaN,rect_b
2,annot_cue_003,NaN,interval,rectangle,3.5,5.50,2.00,NaN,NaN,thoresen_2010_form-building-patterns_p90-91_pa...,NaN,"{""x"": 111, ""y"": 60, ""width"": 57, ""height"": 23}",NaN,rect_c
3,annot_cue_004,NaN,interval,rectangle,34.6,39.80,5.20,NaN,NaN,thoresen_2010_form-building-patterns_p90-91_pa...,NaN,"{""x"": 145, ""y"": 90, ""width"": 160, ""height"": 58}",NaN,rect_a2
4,annot_cue_005,NaN,interval,rectangle,43.5,48.00,4.50,NaN,NaN,thoresen_2010_form-building-patterns_p90-91_pa...,NaN,"{""x"": 385, ""y"": 46, ""width"": 139, ""height"": 20}",NaN,rect_h2
5,annot_cue_006,NaN,interval,rectangle,71.0,75.75,4.75,NaN,NaN,thoresen_2010_form-building-patterns_p90-91_pa...,NaN,"{""x"": 310, ""y"": 93, ""width"": 154, ""height"": 18}",NaN,rect_d3
6,annot_cue_007,NaN,interval,rectangle,76.0,83.50,7.50,NaN,NaN,thoresen_2010_form-building-patterns_p90-91_pa...,NaN,"{""x"": 456, ""y"": 69, ""width"": 229, ""height"": 18}",NaN,rect_b3
7,annot_cue_008,NaN,interval,rectangle,90.5,94.50,4.00,NaN,NaN,thoresen_2010_form-building-patterns_p90-91_pa...,NaN,"{""x"": 14, ""y"": 115, ""width"": 127, ""height"": 31}",NaN,rect_i4
8,annot_cue_009,NaN,interval,rectangle,113.4,116.40,3.00,NaN,NaN,thoresen_2010_form-building-patterns_p90-91_pa...,NaN,"{""x"": 663, ""y"": 82, ""width"": 97, ""height"": 23}",NaN,rect_a4
9,annot_cue_010,NaN,interval,rectangle,121.0,128.50,7.50,NaN,NaN,thoresen_2010_form-building-patterns_p90-91_pa...,NaN,"{""x"": 19, ""y"": 119, ""width"": 251, ""height"": 29}",NaN,rect_i5


### Promoting Columns to Typed Fields with `column_specs`

When a property column should become a typed field, name it in
`column_specs`. The keys are source-column names; the values are
anything the loader can resolve to a field — a bare Python type
(`int` / `float` / `str`) is the simplest form. Here we give two
extra columns explicit types:

In [8]:
class ThoresenTypedLoader(TsvLoader):
    """Selected columns promoted to typed fields."""

    id_column = "event_id"
    start_column = "start_time_sec"
    duration_column = "duration_sec"
    event_type_column = "event_type"
    name_column = "description"

    _default_unit = TimeUnit.seconds
    coordinate_type = NumberType.float

    # Promote these source columns to typed fields.
    column_specs = {
        "image_filename": str,
        "graphical_element_id": str,
    }


typed = ThoresenTypedLoader()
typed.load(THORESEN / "thoresen_test.tsv")
typed.events.to_pandas()

,id,name,temporal_type,event_type,start,end,duration,text_anchor_xy_json,image_filename,alignment_group_id,text_content,rect_coords_json,layer_order,graphical_element_id
0,annot_cue_001,NaN,interval,rectangle,0.0,5.00,5.00,NaN,thoresen_2010_form-building-patterns_p90-91_pa...,NaN,NaN,"{""x"": 10, ""y"": 90, ""width"": 148, ""height"": 55}",NaN,rect_a
1,annot_cue_002,NaN,interval,rectangle,1.5,5.50,4.00,NaN,thoresen_2010_form-building-patterns_p90-91_pa...,NaN,NaN,"{""x"": 40, ""y"": 37, ""width"": 127, ""height"": 21}",NaN,rect_b
2,annot_cue_003,NaN,interval,rectangle,3.5,5.50,2.00,NaN,thoresen_2010_form-building-patterns_p90-91_pa...,NaN,NaN,"{""x"": 111, ""y"": 60, ""width"": 57, ""height"": 23}",NaN,rect_c
3,annot_cue_004,NaN,interval,rectangle,34.6,39.80,5.20,NaN,thoresen_2010_form-building-patterns_p90-91_pa...,NaN,NaN,"{""x"": 145, ""y"": 90, ""width"": 160, ""height"": 58}",NaN,rect_a2
4,annot_cue_005,NaN,interval,rectangle,43.5,48.00,4.50,NaN,thoresen_2010_form-building-patterns_p90-91_pa...,NaN,NaN,"{""x"": 385, ""y"": 46, ""width"": 139, ""height"": 20}",NaN,rect_h2
5,annot_cue_006,NaN,interval,rectangle,71.0,75.75,4.75,NaN,thoresen_2010_form-building-patterns_p90-91_pa...,NaN,NaN,"{""x"": 310, ""y"": 93, ""width"": 154, ""height"": 18}",NaN,rect_d3
6,annot_cue_007,NaN,interval,rectangle,76.0,83.50,7.50,NaN,thoresen_2010_form-building-patterns_p90-91_pa...,NaN,NaN,"{""x"": 456, ""y"": 69, ""width"": 229, ""height"": 18}",NaN,rect_b3
7,annot_cue_008,NaN,interval,rectangle,90.5,94.50,4.00,NaN,thoresen_2010_form-building-patterns_p90-91_pa...,NaN,NaN,"{""x"": 14, ""y"": 115, ""width"": 127, ""height"": 31}",NaN,rect_i4
8,annot_cue_009,NaN,interval,rectangle,113.4,116.40,3.00,NaN,thoresen_2010_form-building-patterns_p90-91_pa...,NaN,NaN,"{""x"": 663, ""y"": 82, ""width"": 97, ""height"": 23}",NaN,rect_a4
9,annot_cue_010,NaN,interval,rectangle,121.0,128.50,7.50,NaN,thoresen_2010_form-building-patterns_p90-91_pa...,NaN,NaN,"{""x"": 19, ""y"": 119, ""width"": 251, ""height"": 29}",NaN,rect_i5


That is the entire idea: **columns are a source artefact; fields are a
Time To Align! artefact.** `column_specs` is the bridge. A bare type is
the gentlest entry — the in-depth CSV/TSV how-to covers the full
resolution chain (composite columns, semantic pitch/id fields, and the
Step-2 `field_specs` promotion stage) for richer formats.

## Nested JSON Column Access with `Field`

The Thoresen data has a `rect_coords_json` column containing pixel
coordinates as JSON:
```json
{"x": 10, "y": 90, "width": 148, "height": 55}
```

Use `Field("column", "nested_field")` to point a coordinate attribute
straight at a nested value. Time To Align! parses the JSON
automatically. `ComputedField` lets you derive a coordinate from a
small formula over those nested values:

In [9]:
from timetoalign.loader import ComputedField, Field  # noqa: E402


class ThoresenGraphicalLoader(TsvLoader):
    """Loader using PIXEL coordinates from a nested JSON column."""

    # Nested fields are addressed directly; JSON is parsed automatically.
    start_column = Field("rect_coords_json", "x")
    end_column = ComputedField(
        "end", formula="rect_coords_json.x + rect_coords_json.width"
    )

    _default_unit = TimeUnit.pixels
    coordinate_type = NumberType.float
    default_event_type = "Rectangle"


graphical = ThoresenGraphicalLoader()
graphical.load(THORESEN / "thoresen_test.tsv")

{
    "unit": str(graphical.unit),
    "coordinate_range": graphical.events.coordinate_range(),
}

{'unit': 'pixels', 'coordinate_range': (10.0, 760.0)}

### Two Coordinate Systems from One File

The same TSV file backs timelines in different coordinate systems —
seconds from the time columns, pixels from the JSON column:

In [10]:
# Physical timeline (seconds)
typed.events.to_pandas()

,id,name,temporal_type,event_type,start,end,duration,text_anchor_xy_json,image_filename,alignment_group_id,text_content,rect_coords_json,layer_order,graphical_element_id
0,annot_cue_001,NaN,interval,rectangle,0.0,5.00,5.00,NaN,thoresen_2010_form-building-patterns_p90-91_pa...,NaN,NaN,"{""x"": 10, ""y"": 90, ""width"": 148, ""height"": 55}",NaN,rect_a
1,annot_cue_002,NaN,interval,rectangle,1.5,5.50,4.00,NaN,thoresen_2010_form-building-patterns_p90-91_pa...,NaN,NaN,"{""x"": 40, ""y"": 37, ""width"": 127, ""height"": 21}",NaN,rect_b
2,annot_cue_003,NaN,interval,rectangle,3.5,5.50,2.00,NaN,thoresen_2010_form-building-patterns_p90-91_pa...,NaN,NaN,"{""x"": 111, ""y"": 60, ""width"": 57, ""height"": 23}",NaN,rect_c
3,annot_cue_004,NaN,interval,rectangle,34.6,39.80,5.20,NaN,thoresen_2010_form-building-patterns_p90-91_pa...,NaN,NaN,"{""x"": 145, ""y"": 90, ""width"": 160, ""height"": 58}",NaN,rect_a2
4,annot_cue_005,NaN,interval,rectangle,43.5,48.00,4.50,NaN,thoresen_2010_form-building-patterns_p90-91_pa...,NaN,NaN,"{""x"": 385, ""y"": 46, ""width"": 139, ""height"": 20}",NaN,rect_h2
5,annot_cue_006,NaN,interval,rectangle,71.0,75.75,4.75,NaN,thoresen_2010_form-building-patterns_p90-91_pa...,NaN,NaN,"{""x"": 310, ""y"": 93, ""width"": 154, ""height"": 18}",NaN,rect_d3
6,annot_cue_007,NaN,interval,rectangle,76.0,83.50,7.50,NaN,thoresen_2010_form-building-patterns_p90-91_pa...,NaN,NaN,"{""x"": 456, ""y"": 69, ""width"": 229, ""height"": 18}",NaN,rect_b3
7,annot_cue_008,NaN,interval,rectangle,90.5,94.50,4.00,NaN,thoresen_2010_form-building-patterns_p90-91_pa...,NaN,NaN,"{""x"": 14, ""y"": 115, ""width"": 127, ""height"": 31}",NaN,rect_i4
8,annot_cue_009,NaN,interval,rectangle,113.4,116.40,3.00,NaN,thoresen_2010_form-building-patterns_p90-91_pa...,NaN,NaN,"{""x"": 663, ""y"": 82, ""width"": 97, ""height"": 23}",NaN,rect_a4
9,annot_cue_010,NaN,interval,rectangle,121.0,128.50,7.50,NaN,thoresen_2010_form-building-patterns_p90-91_pa...,NaN,NaN,"{""x"": 19, ""y"": 119, ""width"": 251, ""height"": 29}",NaN,rect_i5


In [11]:
# Graphical timeline (pixels)
graphical.events.to_pandas()

,id,name,temporal_type,event_type,start,end,duration,text_anchor_xy_json,alignment_group_id,image_filename,text_content,event_id,start_time_sec,duration_sec,graphical_element_id,layer_order,description
0,e000000,NaN,interval,rectangle,10.0,158.0,148.0,NaN,NaN,thoresen_2010_form-building-patterns_p90-91_pa...,NaN,annot_cue_001,0.0,5.00,rect_a,NaN,NaN
1,e000001,NaN,interval,rectangle,40.0,167.0,127.0,NaN,NaN,thoresen_2010_form-building-patterns_p90-91_pa...,NaN,annot_cue_002,1.5,4.00,rect_b,NaN,NaN
2,e000002,NaN,interval,rectangle,111.0,168.0,57.0,NaN,NaN,thoresen_2010_form-building-patterns_p90-91_pa...,NaN,annot_cue_003,3.5,2.00,rect_c,NaN,NaN
3,e000003,NaN,interval,rectangle,145.0,305.0,160.0,NaN,NaN,thoresen_2010_form-building-patterns_p90-91_pa...,NaN,annot_cue_004,34.6,5.20,rect_a2,NaN,NaN
4,e000004,NaN,interval,rectangle,385.0,524.0,139.0,NaN,NaN,thoresen_2010_form-building-patterns_p90-91_pa...,NaN,annot_cue_005,43.5,4.50,rect_h2,NaN,NaN
5,e000005,NaN,interval,rectangle,310.0,464.0,154.0,NaN,NaN,thoresen_2010_form-building-patterns_p90-91_pa...,NaN,annot_cue_006,71.0,4.75,rect_d3,NaN,NaN
6,e000006,NaN,interval,rectangle,456.0,685.0,229.0,NaN,NaN,thoresen_2010_form-building-patterns_p90-91_pa...,NaN,annot_cue_007,76.0,7.50,rect_b3,NaN,NaN
7,e000007,NaN,interval,rectangle,14.0,141.0,127.0,NaN,NaN,thoresen_2010_form-building-patterns_p90-91_pa...,NaN,annot_cue_008,90.5,4.00,rect_i4,NaN,NaN
8,e000008,NaN,interval,rectangle,663.0,760.0,97.0,NaN,NaN,thoresen_2010_form-building-patterns_p90-91_pa...,NaN,annot_cue_009,113.4,3.00,rect_a4,NaN,NaN
9,e000009,NaN,interval,rectangle,19.0,270.0,251.0,NaN,NaN,thoresen_2010_form-building-patterns_p90-91_pa...,NaN,annot_cue_010,121.0,7.50,rect_i5,NaN,NaN


### Creating Timelines

Use `create_timeline()` to turn loaded events into a Timeline object:

In [12]:
# Physical timeline (seconds)
physical_tl = typed.create_timeline(uid="thoresen_physical")
physical_tl

ContinuousPhysicalTimeline(id='thoresen_physical', length=142.5, unit=seconds, events=11, children=0)

In [13]:
physical_tl.get_timestamp_table()

pyarrow.Table
axis: double
thoresen_physical: double
----
axis: [[0,1.5,3.5,5,5.5,...,116.4,121,128.5,141,142.5]]
thoresen_physical: [[0,1.5,3.5,5,5.5,...,116.4,121,128.5,141,142.5]]

In [14]:
# Graphical timeline (pixels)
graphical_tl = graphical.create_timeline(uid="thoresen_graphical")
graphical_tl

DiscreteGraphicalTimeline(id='thoresen_graphical', length=760.0, unit=pixels, events=11, children=0)

**Note:** Both timelines represent the same 11 events in different
coordinate systems:
- **Physical:** `0 - 142.5 seconds` (audio time)
- **Graphical:** `10 - 760 pixels` (image coordinates)

Time To Align! uses these dual representations to align graphical
annotations with audio.

### Child Timelines from Column Values with `group_by`

When your data carries events from **multiple sources** (images, pages,
tracks), pass `group_by` to `create_timeline()` to split the events into
one child timeline per unique value. The Thoresen data has events from
five different image files:

In [15]:
from timetoalign.timelines import create_timeline  # noqa: E402

grouped_tl = create_timeline(typed, group_by="image_filename")
grouped_tl

ContinuousPhysicalTimeline(id='tl:1', length=142.5, unit=seconds, events=0, children=5)

In [16]:
# Each child timeline represents events from one image
{
    "parent_id": grouped_tl.id,
    "n_children": grouped_tl.n_children,
    "children": {
        child.id: len(child._events) if child._events else 0
        for _, child in grouped_tl.iter_children()
    },
}

{'parent_id': 'tl:1',
 'n_children': 5,
 'children': {'thoresen_2010_form-building-patterns_p90-91_page1_1.jpeg': 3,
  'thoresen_2010_form-building-patterns_p90-91_page1_2.jpeg': 2,
  'thoresen_2010_form-building-patterns_p90-91_page1_3.jpeg': 2,
  'thoresen_2010_form-building-patterns_p90-91_page1_4.jpeg': 2,
  'thoresen_2010_form-building-patterns_p90-91_page2_1.jpeg': 2}}

***

## Summary

| Goal | How |
|------|-----|
| Load a known TSV/CSV format | `Ms3Loader()` (or another built-in) + `loader.load(path)` |
| Map your own columns | Subclass `TsvLoader` / `CsvLoader`, set `start_column` etc. |
| Promote a column to a typed field | Name it in `column_specs` (`{"col": int}`) |
| Reach a nested JSON value | `Field("column", "nested")` as a coordinate attribute |
| Derive a coordinate | `ComputedField("end", formula="...")` |
| Split into child timelines | `create_timeline(loader, group_by="column")` |

```python
from timetoalign.loader.tabular import TsvLoader

class MyLoader(TsvLoader):
    start_column = "onset"
    duration_column = "dur"
    column_specs = {"pitch": int, "velocity": int}
```

> **Key Takeaway:** Tabular loaders map CSV/TSV columns to Time To Align!
> events declaratively. Unnamed columns ride along as property columns;
> `column_specs` promotes the ones you want as typed fields; `Field`
> reaches nested JSON. For the full column-to-field mechanism — composite
> columns, semantic field types, and Step-2 promotion — continue to the
> in-depth CSV/TSV how-to.